In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pickle
from tqdm import tqdm
from torchvision import transforms
import os

from HARUnet_model import HARU_net

#from torch.cuda.amp import GradScaler, autocast
from torchmetrics.image.ssim import StructuralSimilarityIndexMeasure as ssim
from utils import psnr, batch_psnr

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
class EarlyStopping:
    def __init__(self, patience=5, delta=0):
        """
        Args:
            patience (int): How many epochs to wait after last improvement.
            delta (float): Minimum change to consider an improvement.
        """
        self.patience = patience
        self.delta = delta
        self.best_loss = None
        self.counter = 0
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            print(f'Best loss = {self.best_loss}')
            self.counter = 0

In [3]:
# Load data
def load_data(pickle_file):
    with open(pickle_file, 'rb') as f:
        patches = pickle.load(f)
    return patches  # Expecting a NumPy array (N, 1, H, W)

# Compute BM3D targets
#def apply_bm3d(patches, sigma):
#    return np.array([bm3d(patch, sigma) for patch in patches])

class CBCTDataset(Dataset):
    def __init__(self, noisy_patches, target_patches):
        self.noisy = noisy_patches  # Keep as is
        self.target = target_patches  # Keep as is

    def __len__(self):
        return len(self.noisy)

    def __getitem__(self, idx):
        noisy_tensor = torch.tensor(self.noisy[idx], dtype=torch.float32)
        target_tensor = torch.tensor(self.target[idx], dtype=torch.float32)
        return noisy_tensor, target_tensor

In [ ]:
#data_dir = r"D:\CBCT_Denoising\Pickled CBCT Data"

pickled_train_inputs = "Give path to the noisy samples to be used as inputs for training" # Code for loading and noising data will be updated in the next version
pickled_train_targets = r"Give path to the clean out labels for training"
train_inputs = load_data(pickled_train_inputs)  # Shape: (N, 1, H, W)
train_targets = load_data(pickled_train_targets)  # Shape: (N, 1, H, W)

train_dataset = CBCTDataset(train_inputs, train_targets)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)

In [ ]:
pickled_val_inputs = "Give path to the noisy samples to be used as inputs for training" # Code for loading and noising data will be updated in the next version
pickled_val_targets = r"Give path to the clean out labels for training"
val_inputs = load_data(pickled_val_inputs)  # Shape: (N, 1, H, W)
val_targets = load_data(pickled_val_targets)  # Shape: (N, 1, H, W)

val_dataset = CBCTDataset(val_inputs, val_targets)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=True, num_workers=0, pin_memory=True)

savedir = r"D:\CBCT_Denoising\Pickled CBCT Data"
pickle_file_train = "train_CBCT_patches.pkl"
train_patches = load_data(os.path.join(data_dir,pickle_file_train))  # Shape: (N, 1, H, W)
train_inputs = train_patches
sigma = 0.03
train_targets = apply_bm3d(train_inputs, sigma)


save_train_inputs = os.path.join(savedir,f"Train_CBCT_inputs.pkl")
with open(save_train_inputs, "wb") as f:
    pickle.dump(train_inputs, f)

save_train_tragets = os.path.join(savedir,f"Train_CBCT_targets_bm3d_{sigma}.pkl")
with open(save_train_tragets, "wb") as f:
    pickle.dump(train_targets, f)

pickle_file_val = "Val_CBCT_patches.pkl"
val_patches = load_data(os.path.join(data_dir,pickle_file_val))  # Shape: (N, 1, H, W)
val_inputs = val_patches
val_targets = apply_bm3d(val_inputs, sigma)


save_val_inputs = os.path.join(savedir,f"Val_CBCT_inputs.pkl")
with open(save_val_inputs, "wb") as f:
    pickle.dump(val_inputs, f)

save_val_tragets = os.path.join(savedir,f"Val_CBCT_targets_bm3d_{sigma}.pkl")
with open(save_val_tragets, "wb") as f:
    pickle.dump(val_targets, f)

In [6]:
model = HARU_net(dim=64, hab_heads=8, hab_ws=16, hab_mlp_ratio=2.0, hab_depth=6)
model = model.to(device)
#model = model.to(device)
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs!")
    model = torch.nn.DataParallel(model)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=5,
    threshold=1e-4,      # consider changes smaller than this as "no improvement"
    threshold_mode='rel',
    cooldown=0,
    min_lr=1e-8,
    verbose=True
)


Using 3 GPUs!


c:\Users\au711969\AppData\Local\anaconda3\envs\KnTorch\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


In [ ]:
save_dir = "Give path to the directory for saving the trained model" # Code for loading and noising data will be updated in the next version
folder_name = f'HARUnetv2_epoch_models' # Subdirectory for saving the trained models
folder_path = os.path.join(save_dir,folder_name)
os.makedirs(folder_path, exist_ok=True)

In [ ]:
early_stopping = EarlyStopping(patience=20, delta=1e-5)
    
train_loss_history = []
train_inpsnr_history = []
train_outpsnr_history = []
train_BM3Drltvpsnr_history = []

val_loss_history = []
val_inpsnr_history = []
val_outpsnr_history = []
val_BM3Drltvpsnr_history = []

epochs = 1000
for epoch in range(epochs):
    model.train()
    train_loss = 0.0
    train_inpsnr = 0
    train_outpsnr = 0
    train_BM3Drltvpsnr = 0
    train_loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    for train_inputs, train_targets in train_loop:
        train_inputs, train_targets = train_inputs.unsqueeze(1).to(device), train_targets.unsqueeze(1).to(device)

        optimizer.zero_grad()
        train_outputs = model(train_inputs)
    
        batch_loss = criterion(train_outputs, train_targets)
        batch_loss.backward()
        optimizer.step()  

        train_loss += batch_loss.item()
        train_inpsnr += batch_psnr(train_inputs, train_targets)
        train_outpsnr += batch_psnr(train_outputs, train_targets)

    train_loss /= len(train_loader)
    epoch_train_inpsnr = train_inpsnr / len(train_loader)
    epoch_train_outpsnr = train_outpsnr / len(train_loader)

    train_loss_history.append(train_loss)
    train_inpsnr_history.append(epoch_train_inpsnr)
    train_outpsnr_history.append(epoch_train_outpsnr)

    model.eval()
    val_loss = 0.0
    val_inpsnr = 0
    val_outpsnr = 0
    val_BM3Drltvpsnr = 0
    count = 0
    for val_inputs, val_target in val_loader:

        val_inputs, val_targets = val_inputs.unsqueeze(1).to(device), val_target.unsqueeze(1).to(device)
        optimizer.zero_grad()

        with torch.no_grad():
            val_outputs = model(val_inputs)
        if batch_psnr(val_inputs, val_targets) != float('inf'):
            val_loss += criterion(val_outputs, val_targets).item()
            val_inpsnr += batch_psnr(val_inputs, val_targets)
            val_outpsnr += batch_psnr(val_outputs, val_targets)

        else:
            count = count + 1

    val_loss /= (len(val_loader)-count)
    epoch_val_inpsnr = val_inpsnr / (len(val_loader)-count)
    epoch_val_outpsnr = val_outpsnr / (len(val_loader)-count)
    epoch_val_BM3Drltvpsnr = val_BM3Drltvpsnr / (len(val_loader)-count)
    
    scheduler.step(val_loss)
    val_loss_history.append(val_loss)
    val_inpsnr_history.append(epoch_val_inpsnr)
    val_outpsnr_history.append(epoch_val_outpsnr)

    train_loop.set_postfix(train_loss=train_loss)
    torch.save(model.state_dict(), os.path.join(folder_path,f"HARUunet_state_dict_{epoch}.pth"))
    

    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch [{epoch+1}/{epochs}], Train loss: {train_loss:.8f}, Train input PSNR: {epoch_train_inpsnr:.3f}, Train outPSNR: {epoch_train_outpsnr:.3f}') 
    print(f'LR    [{current_lr}],, Validation loss: {val_loss:.8f}, Validation input PSNR: {epoch_val_inpsnr:.3f}, Validation outPSNR: {epoch_val_outpsnr:.3f}')

    early_stopping(val_loss)
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        total_training_epochs = epoch+1
        break

In [ ]:
torch.save(model, os.path.join(folder_path,f'HARUnet_trainedon_CadavarData_epoch_nr{epoch}.pth'))


savedir = r"D:\CBCT_Denoising\Pickled CBCT Data"
data_dir_ = r"O:\HE_IOOS-Khuram\CBCT data cadavers\Pickled CBCT Data"
pickle_file_test = "Test_CBCT_patches.pkl"
test_inputs = load_data(os.path.join(data_dir_,pickle_file_test))  # Shape: (N, 1, H, W)
sigma = 0.03
test_targets = apply_bm3d(test_inputs, sigma=0.03)

savedir = r"D:\CBCT_Denoising\Pickled CBCT Data"
data_dir_ = r"O:\HE_IOOS-Khuram\CBCT data cadavers\Pickled CBCT Data"
pickle_file_test = "Test_CBCT_patches.pkl"
test_inputs = load_data(os.path.join(data_dir_,pickle_file_test))  # Shape: (N, 1, H, W)
sigma = 0.03
test_targets = apply_bm3d(test_inputs, sigma=0.03)

test_savefilename = os.path.join(savedir,f"Test_CBCT_inputs.pkl")
with open(test_savefilename, "wb") as f:
    pickle.dump(test_inputs, f)

test_savefilename = os.path.join(savedir,f"Test_CBCT_targets_bm3d_{sigma}.pkl")
with open(test_savefilename, "wb") as f:
    pickle.dump(test_inputs, f)



In [ ]:
pickled_test_inputs = "Give path to the noisy samples to be used as inputs for training" # Code for loading and noising data will be updated in the next version
pickled_test_targets = "Give path to the clean out labels for training"
test_inputs = load_data(pickled_test_inputs)  # Shape: (N, 1, H, W)
test_targets = load_data(pickled_test_targets)  # Shape: (N, 1, H, W)
test_dataset = CBCTDataset(test_inputs, test_targets)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=True, num_workers=0,pin_memory=True)

In [ ]:
model.eval()
test_loss = 0.0
test_inpsnr = 0
test_outpsnr = 0
test_BM3Drltvpsnr = 0
for test_inputs, test_targets in test_loader:

    test_inputs, test_targets = test_inputs.unsqueeze(1).to(device), test_targets.unsqueeze(1).to(device)
    optimizer.zero_grad()
            
    with torch.no_grad():
        test_outputs = model(test_inputs)
    if batch_psnr(test_inputs, test_targets) != float('inf'):
        test_loss += criterion(test_outputs, test_targets).item()
        test_inpsnr += batch_psnr(test_inputs, test_targets)
        print(f'Batch_psnr = {batch_psnr(test_inputs, test_targets)}, total test_inpsnr = {test_inpsnr}')
        test_outpsnr += batch_psnr(test_outputs, test_targets)

test_loss /= (len(test_loader)-1)
epoch_test_inpsnr = test_inpsnr /(len(test_loader)-1)
epoch_test_outpsnr = test_outpsnr / (len(test_loader)-1)

print(f'............, Test loss: {test_loss:.8f}, Test Target PSNR: {epoch_test_inpsnr:.3f}, Test outPSNR: {epoch_test_outpsnr:.3f}')
